In [ ]:
# --- cell 1 · bootstrap（任务书 §0.1 末、§0.2）—— 五个 B notebook 里这一格完全相同 ---
import pathlib, sys

# (1) 向上搜索含 npnn/ 的目录 —— 不依赖 Jupyter 的启动目录
def _find_taskb_root(start):
    for p in (start, *start.parents):
        if (p / 'npnn').is_dir():
            return p
        if (p / 'task_b' / 'npnn').is_dir():      # 从项目根启动 Jupyter 的情况
            return p / 'task_b'
    return None

_here = pathlib.Path.cwd().resolve()
TASKB_ROOT = _find_taskb_root(_here)
assert TASKB_ROOT is not None, f'从 {_here} 逐级往上找不到含 npnn/ 的目录；请从 task_b/ 或项目根启动'
sys.path.insert(0, str(TASKB_ROOT))

# (2) 物理挡住「误 import A 的 src」：把项目根从 sys.path 里摘掉（任务书 §0.1.2）
_root = TASKB_ROOT.parent
sys.path[:] = [e for e in sys.path if pathlib.Path(e or '.').resolve() != _root]

# (3) 线程预算：必须在 import numpy 之前（BLAS 首次 import 时锁死线程数）
import config
config.limit_threads()

# (4) 环境留痕 —— 报告 B.1 要用：哪个解释器、哪个 numpy
import numpy as np
print('sys.executable:', sys.executable)
print('sys.prefix    :', sys.prefix)
print('numpy         :', np.__version__)
print('TASKB_ROOT    :', TASKB_ROOT)


# 12 · 与 PyTorch 整网对拍（L2 —— 10 分的核心证据）

对应任务书：§3.1（L2 比 L3 重要得多）、§3.2（严格协议）。

> L3 的「精度相近」是弱证据（lr 抖动、随机性都能造成几个点差异）；
> L2 的「loss 轨迹逐位重合」是强证据 —— 它证明整个前向 + 反向都对。

## §3.2 协议（照着做）
| 环节 | 做法 | 坑 |
|---|---|---|
| 架构 | 定义**一次**，两边用同一份配置生成 | 别一边 3 层一边 4 层 |
| 权重 | numpy 侧生成 → `torch.from_numpy(w.copy())`，`no_grad` 写入 | dtype / 形状要一致；`copy()` 避免共享内存 |
| 数据 | 同一批 uint8 图像，两边各自归一化 | 归一化公式（均值/标准差/除 255 的顺序）必须一致 |
| 优化器 | 你的 Adam vs `torch.optim.Adam`，同 lr / betas / eps | 见 §2.4 四坑 |
| 设备 | **全程 CPU + float64**，再用 float32 确认误差量级 | GPU/cuDNN 非确定性会掩盖真 bug |
| 判据 | N 步后 loss 相对差 <1e-8（f64）/ <1e-4（f32） | 分叉 → 二分定位：第几步开始，就是哪一步的嫌疑 |

✍️ 本 notebook 的产出：
- [ ] 两条 loss 曲线重叠画在同一张图（肉眼几乎分不开）
- [ ] `|loss_np − loss_torch|` vs step，**对数纵轴** —— 比「两条线重合」信息量大得多
- [ ] 若 N 步不分叉：最终参数差值直方图
- [ ] 若分叉：把定位过程写进报告（「第 k 步开始分叉 → 发现 xxx」这种技术叙事最打动人）

⛔ 本 notebook 可以 import torch —— 它是验证工具，不是交付实现（报告里要明确区分）。
